<a href="https://colab.research.google.com/github/Defined0101/data/blob/FRS-78-Semisupervised-model/MergedDataset/Bert.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
import nltk
import re
import string
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import classification_report, accuracy_score
import gc

import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# Parquet dosyasının yolu
data_path = '/content/drive/MyDrive/Defined 0101/Data/CategorizedData/foodcom_categorized_v2.parquet'  # Buraya kendi dosya yolunuzu yazın

# Parquet dosyasını yükle
df = pd.read_parquet(data_path)

In [4]:
df.MainCategory.value_counts()

,count
MainCategory,
Other,822035
Dessert,256315
Main Dish,197506
Side Dishes,129704
Breakfast,121075
Appetizer,31202
Beverages,29759
Soup,14257


In [ ]:
# İlk 5 satırı görüntüle
print(df.head())

# Veri boyutu
print(f"Veri seti boyutu: {df.shape}")

# Kategorilerin dağılımı
print(df['MainCategory'].value_counts())


                                    name  \
0            Grilled Garlic Cheese Grits   
1  Simple Shrimp and Andouille Jambalaya   
2             black-and-white bean salad   
3             Crock Pot Italian Zucchini   
4             Crock Pot Italian Zucchini   

                                        instructions  \
0  1. I a sauce pan, bring water to a boil; slowl...   
1  1. In a food processor, pulse the onion, red p...   
2  1. In a large bowl, combine beans, tomato, oni...   
3  1. Put all ingredients in the crock pot and co...   
4  1. Spray a slow cooker container with olive oi...   

                                         ingredients  total_time  calories  \
0  [{'name': 'water', 'quantity': 4.0, 'unit': 'c...        65.0     144.8   
1                                                 []        45.0     756.5   
2  [{'name': 'white beans', 'quantity': 1.0, 'uni...         5.0     159.0   
3                                                 []       370.0      47.1   
4  [{'na

In [ ]:
# Eksik değerlerin sayısı
print(df.isnull().sum())

# MainCategory sütunundaki eksik değerlerin oranı
missing_categories = df['MainCategory'].isnull().sum()
total_samples = len(df)
print(f"Etiketi olmayan tarif sayısı: {missing_categories}")
print(f"Toplam tarif sayısı: {total_samples}")
print(f"Etiketi olmayan tariflerin oranı: {missing_categories / total_samples * 100:.2f}%")


name                 0
instructions         5
ingredients          0
total_time      136153
calories             0
fat                  0
protein              0
carbohydrate         0
desc                 5
MainCategory         0
dtype: int64
Etiketi olmayan tarif sayısı: 0
Toplam tarif sayısı: 1601853
Etiketi olmayan tariflerin oranı: 0.00%


In [ ]:
# nltk paketindeki 'stopwords' listesini kullanmak için indirme işlemi
nltk.download('stopwords')
from nltk.corpus import stopwords

# Önce stopwords listesini ve diğer sabitleri tanımlayalım
stop_words = set(stopwords.words('english'))
stop_words_regex = r'\b(' + r'|'.join(map(re.escape, stop_words)) + r')\b'
punctuation_regex = '[' + re.escape(string.punctuation) + ']'

# Metin ön işleme fonksiyonları
def clean_text_series(series):
    series = series.fillna('').astype(str)
    series = series.str.lower()
    series = series.str.replace(punctuation_regex, '', regex=True)
    series = series.str.replace(r'\d+', '', regex=True)
    series = series.str.replace(stop_words_regex, '', regex=True)
    series = series.str.replace(r'\s+', ' ', regex=True)
    return series.str.strip()


[nltk_data] Downloading package stopwords to C:\Users\Sami
[nltk_data]     Berkan\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
# 'ingredients' listesini metin haline getirme ve temizleme
def ingredients_to_text_series(ingredients_series):
    ingredient_names = ingredients_series.apply(
        lambda x: ' '.join([ingredient.get('name', '') for ingredient in x]) if isinstance(x, list) else ''
    )
    return clean_text_series(ingredient_names)

# 'instructions' sütununu temizleme
df['clean_instructions'] = clean_text_series(df['instructions'])

# 'desc' sütununu temizleme
df['clean_desc'] = clean_text_series(df['desc'])

# 'ingredients' sütununu temizleme
df['clean_ingredients'] = ingredients_to_text_series(df['ingredients'])


In [ ]:
df['Result'] = np.nan

nan_indices = df.sample(frac=0.3, random_state=42).index
df.loc[nan_indices, 'Result'] = df.loc[nan_indices, 'MainCategory']
df.loc[nan_indices, 'MainCategory'] = np.nan

C:\Users\Sami Berkan\AppData\Local\Temp\ipykernel_29820\2200394735.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['Beverages' 'Other' 'Breads' ... 'Side Dishes' 'Side Dishes' 'Dessert']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[nan_indices, 'Result'] = df.loc[nan_indices, 'MainCategory']


In [ ]:
# Etiketli veriler
labeled_data = df[df['MainCategory'].notnull()].copy()

# Etiketsiz veriler
unlabeled_data = df[df['MainCategory'].isnull()].copy()


In [ ]:
# Özelliklerin birleştirilmesi
labeled_data['text'] = labeled_data['clean_instructions'] + ' ' + labeled_data['clean_desc'] + ' ' + labeled_data['clean_ingredients']
unlabeled_data['text'] = unlabeled_data['clean_instructions'] + ' ' + unlabeled_data['clean_desc'] + ' ' + unlabeled_data['clean_ingredients']


In [ ]:
# GPU kullanılabilirliğini kontrol edin
print("GPU Mevcut mu:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Adı:", torch.cuda.get_device_name(0))
else:
    print("GPU kullanılamıyor. CPU kullanılacak.")

In [ ]:
# Eğitim ve doğrulama verilerini ayırma
train_texts, val_texts, train_labels, val_labels = train_test_split(
    labeled_data['text'], labeled_data['MainCategory'], test_size=0.2, random_state=42
)

# Kategorik etiketleri kodlama
label_encoder = LabelEncoder()
train_labels = label_encoder.fit_transform(train_labels)
val_labels = label_encoder.transform(val_labels)

In [ ]:
# Tokenizer yükleme
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

# Veriyi tokenize etme
train_encodings = tokenizer(list(train_texts), truncation=True, padding=True, max_length=128)
val_encodings = tokenizer(list(val_texts), truncation=True, padding=True, max_length=128)


In [ ]:
# Dataset oluşturma
class RecipeDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

train_dataset = RecipeDataset(train_encodings, train_labels)
val_dataset = RecipeDataset(val_encodings, val_labels)

In [ ]:
# Modeli GPU'ya yükleyin
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=len(label_encoder.classes_)
).to(device)

# Eğitim ayarları
training_args = TrainingArguments(
    output_dir='/content/drive/MyDrive/GradProject/results',  # Sonuçları Google Drive'a kaydetmek için
    evaluation_strategy="epoch",                 # Her epoch sonunda değerlendirme
    per_device_train_batch_size=16,              # GPU için uygun batch boyutu
    per_device_eval_batch_size=16,
    num_train_epochs=3,                          # Eğitim epoch sayısı
    save_strategy="epoch",                       # Her epoch sonunda modeli kaydet
    logging_dir='/content/drive/MyDrive/GradProject/logs',   # Loglar için Drive dizini
    logging_steps=10,                            # Loglama sıklığı
    report_to="none",                            # WandB veya başka bir platforma rapor göndermeyi devre dışı bırak
)

# Trainer nesnesi
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

In [ ]:
# Model eğitimi
trainer.train()

# Modeli ve tokenizer'ı kaydet
model.save_pretrained('/content/drive/MyDrive/GradProject/bert_model')
tokenizer.save_pretrained('/content/drive/MyDrive/GradProject/bert_model')